# 🔬 Visual Reasoning Pipeline - Multi-Architecture Analysis

**Complete End-to-End Pipeline in One Cell**

This notebook implements an end-to-end experimental pipeline combining:
- **Vision Transformers** (ViT, Swin) for supervised learning
- **CLIP** for vision-language zero-shot reasoning
- **GANs** for generative modeling
- **Cross-architectural analysis** for embedding alignment and failure discovery

## Pipeline Stages

1. **Stage 1**: Transformer-based visual understanding (ViT, Swin)
2. **Stage 2**: Vision-language reasoning with CLIP
3. **Stage 3**: Cross-architectural embedding alignment
4. **Stage 4**: Image generation with GANs
5. **Stage 5**: Stress testing models with generated data
6. **Stage 6**: Failure discovery and analysis

## Dataset Requirements

The notebook expects a dataset at `/kaggle/input/fabric-final-dataset/FabrIQ_Final_Dataset/` with structure:
```
FabrIQ_Final_Dataset/
└── train/
    └── images/
        ├── FabrIQ_class_name_00001.jpg
        └── ...
```

**Note**: This pipeline is designed for small datasets (15-20 images) to demonstrate multi-architecture analysis in low-data environments.

---

## 🚀 Run All Stages Below


## Complete Pipeline - Run This Cell


In [ ]:
# ============================================================================
# COMPLETE VISUAL REASONING PIPELINE - ALL STAGES IN ONE CELL
# ============================================================================

# Install required packages
%pip install timm -q
%pip install git+https://github.com/openai/CLIP.git -q
%pip install ftfy regex -q

import os
import json
import time
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import timm
import clip
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# SETUP
# ============================================================================
print("="*80)
print("🔬 VISUAL REASONING PIPELINE - INITIALIZATION")
print("="*80)

# Kaggle paths
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING / 'visual_reasoning_output'
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'failure_cases').mkdir(exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
print(f"✅ Output directory: {OUTPUT_DIR}")

# Class definitions
CLASSES = [
    'contamination', 'selvet', 'gray_stitch', 'cut',
    'baekra', 'color_issue', 'stain'
]

NUM_SAMPLES = 20  # Use 15-20 images as specified

# Dataset class
class FabricDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx], self.image_paths[idx]
        except Exception as e:
            print(f"Error loading {self.image_paths[idx]}: {e}")
            return torch.zeros(3, 224, 224), self.labels[idx], self.image_paths[idx]

# ============================================================================
# LOAD DATASET
# ============================================================================
print("\n" + "="*80)
print("LOADING DATASET")
print("="*80)

DATASET_PATH = KAGGLE_INPUT / 'fabric-final-dataset' / 'FabrIQ_Final_Dataset'
train_dir = DATASET_PATH / 'train' / 'images'

if not train_dir.exists():
    DATASET_PATH = KAGGLE_INPUT / 'fabric' / 'FabrIQ_Final_Dataset'
    train_dir = DATASET_PATH / 'train' / 'images'

print(f"📁 Dataset path: {DATASET_PATH}")

image_paths = []
labels = []
all_images = list(train_dir.glob("*.jpg"))[:NUM_SAMPLES]

for img_path in all_images:
    parts = img_path.stem.split('_')
    if len(parts) >= 2 and parts[0] == 'FabrIQ':
        class_parts = parts[1:-1]
        class_name = ' '.join(class_parts)
        if class_name in CLASSES:
            image_paths.append(str(img_path))
            labels.append(CLASSES.index(class_name))

print(f"✅ Loaded {len(image_paths)} images")

# ============================================================================
# STAGE 1: VISION TRANSFORMERS
# ============================================================================
print("\n" + "="*80)
print("STAGE 1: Vision Transformer-Based Visual Understanding")
print("="*80)

# Train ViT
print("   Training ViT...")
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=len(CLASSES))
vit_model = vit_model.to(device)

if len(image_paths) >= 4:
    dataset = FabricDataset(image_paths, labels)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(vit_model.parameters(), lr=1e-4, weight_decay=0.01)
    vit_model.train()
    for epoch in range(2):
        for images, labels_batch, _ in tqdm(dataloader, desc=f"ViT Epoch {epoch+1}/2"):
            images, labels_batch = images.to(device), labels_batch.to(device)
            optimizer.zero_grad()
            outputs = vit_model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

# Evaluate ViT
vit_model.eval()
vit_predictions, vit_confidences, vit_inference_times = [], [], []
correct = 0
dataset = FabricDataset(image_paths, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

with torch.no_grad():
    for images, labels_batch, _ in dataloader:
        images, labels_batch = images.to(device), labels_batch.to(device)
        start_time = time.time()
        outputs = vit_model(images)
        vit_inference_times.append(time.time() - start_time)
        probs = torch.softmax(outputs, dim=1)
        conf, preds = torch.max(probs, 1)
        vit_predictions.extend(preds.cpu().numpy())
        vit_confidences.extend(conf.cpu().numpy())
        correct += (preds == labels_batch).sum().item()

vit_accuracy = correct / len(image_paths) if len(image_paths) > 0 else 0
vit_avg_confidence = np.mean(vit_confidences)
vit_avg_inference_time = np.mean(vit_inference_times)

# Extract ViT embeddings
vit_embeddings = []
with torch.no_grad():
    for images, _, _ in dataloader:
        images = images.to(device)
        try:
            features = vit_model.forward_features(images)
            if len(features.shape) == 4:
                features = features.mean(dim=[2, 3])
            elif len(features.shape) == 3:
                features = features.mean(dim=1)
            vit_embeddings.append(features.cpu().numpy())
        except:
            vit_embeddings.append(vit_model(images).cpu().numpy())
vit_embeddings = np.concatenate(vit_embeddings, axis=0) if vit_embeddings else np.zeros((len(image_paths), 768))

vit_results = {
    'model_name': 'ViT', 'accuracy': vit_accuracy, 'avg_confidence': float(vit_avg_confidence),
    'avg_inference_time': float(vit_avg_inference_time), 'predictions': vit_predictions,
    'confidences': [float(c) for c in vit_confidences]
}
with open(OUTPUT_DIR / 'vit_results.json', 'w') as f:
    json.dump(vit_results, f, indent=2)
print(f"✅ ViT Accuracy: {vit_accuracy:.4f}")

# Train Swin
print("\n   Training Swin Transformer...")
swin_model = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=len(CLASSES))
swin_model = swin_model.to(device)

if len(image_paths) >= 4:
    dataset = FabricDataset(image_paths, labels)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(swin_model.parameters(), lr=1e-4, weight_decay=0.01)
    swin_model.train()
    for epoch in range(2):
        for images, labels_batch, _ in tqdm(dataloader, desc=f"Swin Epoch {epoch+1}/2"):
            images, labels_batch = images.to(device), labels_batch.to(device)
            optimizer.zero_grad()
            outputs = swin_model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

# Evaluate Swin
swin_model.eval()
swin_predictions, swin_confidences, swin_inference_times = [], [], []
correct = 0
dataset = FabricDataset(image_paths, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

with torch.no_grad():
    for images, labels_batch, _ in dataloader:
        images, labels_batch = images.to(device), labels_batch.to(device)
        start_time = time.time()
        outputs = swin_model(images)
        swin_inference_times.append(time.time() - start_time)
        probs = torch.softmax(outputs, dim=1)
        conf, preds = torch.max(probs, 1)
        swin_predictions.extend(preds.cpu().numpy())
        swin_confidences.extend(conf.cpu().numpy())
        correct += (preds == labels_batch).sum().item()

swin_accuracy = correct / len(image_paths) if len(image_paths) > 0 else 0
swin_avg_confidence = np.mean(swin_confidences)
swin_avg_inference_time = np.mean(swin_inference_times)

# Extract Swin embeddings
swin_embeddings = []
with torch.no_grad():
    for images, _, _ in dataloader:
        images = images.to(device)
        try:
            features = swin_model.forward_features(images)
            if len(features.shape) == 4:
                features = features.mean(dim=[2, 3])
            elif len(features.shape) == 3:
                features = features.mean(dim=1)
            swin_embeddings.append(features.cpu().numpy())
        except:
            swin_embeddings.append(swin_model(images).cpu().numpy())
swin_embeddings = np.concatenate(swin_embeddings, axis=0) if swin_embeddings else np.zeros((len(image_paths), 768))

swin_results = {
    'model_name': 'Swin', 'accuracy': swin_accuracy, 'avg_confidence': float(swin_avg_confidence),
    'avg_inference_time': float(swin_avg_inference_time), 'predictions': swin_predictions,
    'confidences': [float(c) for c in swin_confidences]
}
with open(OUTPUT_DIR / 'swin_results.json', 'w') as f:
    json.dump(swin_results, f, indent=2)
print(f"✅ Swin Accuracy: {swin_accuracy:.4f}")

# ============================================================================
# STAGE 2: CLIP VISION-LANGUAGE REASONING
# ============================================================================
print("\n" + "="*80)
print("STAGE 2: Vision-Language Reasoning with CLIP")
print("="*80)

print("   Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

text_descriptions = [f"a photo of {cls}" for cls in CLASSES]
text_tokens = clip.tokenize(text_descriptions).to(device)

with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

clip_predictions, clip_confidences, clip_embeddings = [], [], []
correct = 0

with torch.no_grad():
    for i, img_path in enumerate(tqdm(image_paths, desc="   CLIP classification")):
        try:
            image = Image.open(img_path).convert('RGB')
            image_tensor = clip_preprocess(image).unsqueeze(0).to(device)
            image_features = clip_model.encode_image(image_tensor)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            clip_embeddings.append(image_features.cpu().numpy())
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            conf, pred = similarity.max(dim=-1)
            clip_predictions.append(pred.item())
            clip_confidences.append(conf.item())
            if pred.item() == labels[i]:
                correct += 1
        except Exception as e:
            clip_predictions.append(0)
            clip_confidences.append(0.0)
            clip_embeddings.append(np.zeros((1, 512)))

clip_embeddings = np.concatenate(clip_embeddings, axis=0) if clip_embeddings else np.zeros((len(image_paths), 512))
clip_accuracy = correct / len(image_paths) if len(image_paths) > 0 else 0

clip_success_supervised_failed = 0
supervised_success_clip_failed = 0
for i in range(len(image_paths)):
    clip_correct = clip_predictions[i] == labels[i]
    vit_correct = vit_predictions[i] == labels[i]
    swin_correct = swin_predictions[i] == labels[i]
    supervised_correct = vit_correct or swin_correct
    if clip_correct and not supervised_correct:
        clip_success_supervised_failed += 1
    if supervised_correct and not clip_correct:
        supervised_success_clip_failed += 1

clip_results = {
    'zero_shot_accuracy': clip_accuracy, 'zero_shot_predictions': clip_predictions,
    'zero_shot_confidences': [float(c) for c in clip_confidences],
    'clip_success_count': clip_success_supervised_failed,
    'supervised_success_count': supervised_success_clip_failed
}
with open(OUTPUT_DIR / 'clip_results.json', 'w') as f:
    json.dump(clip_results, f, indent=2)
print(f"✅ CLIP Zero-shot Accuracy: {clip_accuracy:.4f}")

# ============================================================================
# STAGE 3: EMBEDDING ALIGNMENT
# ============================================================================
print("\n" + "="*80)
print("STAGE 3: Cross-Architectural Embedding Alignment")
print("="*80)

vit_emb_norm = vit_embeddings / (np.linalg.norm(vit_embeddings, axis=1, keepdims=True) + 1e-8)
swin_emb_norm = swin_embeddings / (np.linalg.norm(swin_embeddings, axis=1, keepdims=True) + 1e-8)
clip_emb_norm = clip_embeddings / (np.linalg.norm(clip_embeddings, axis=1, keepdims=True) + 1e-8)

all_embeddings = np.concatenate([vit_emb_norm, swin_emb_norm, clip_emb_norm], axis=0)
embedding_names = ['vit'] * len(vit_emb_norm) + ['swin'] * len(swin_emb_norm) + ['clip'] * len(clip_emb_norm)

print("   Projecting embeddings to 2D...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_embeddings)-1))
projected = tsne.fit_transform(all_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax1 = axes[0]
colors = {'vit': 'red', 'swin': 'blue', 'clip': 'green'}
for model_name in ['vit', 'swin', 'clip']:
    indices = [i for i, name in enumerate(embedding_names) if name == model_name]
    if indices:
        ax1.scatter(projected[indices, 0], projected[indices, 1], c=colors.get(model_name, 'gray'),
                   label=model_name.upper(), alpha=0.6, s=50)
ax1.set_title('Embeddings by Model Architecture')
ax1.set_xlabel('Dimension 1')
ax1.set_ylabel('Dimension 2')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
unique_labels = list(set(labels))
colors_map = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
for i, label in enumerate(unique_labels):
    label_indices = []
    for img_idx, img_label in enumerate(labels):
        if img_label == label:
            for model_name in ['vit', 'swin', 'clip']:
                model_start = sum(len(vit_emb_norm) if m == 'vit' else len(swin_emb_norm) if m == 'swin' else len(clip_emb_norm)
                                for m in ['vit', 'swin', 'clip'] if m < model_name)
                emb_idx = model_start + img_idx
                if emb_idx < len(projected):
                    label_indices.append(emb_idx)
    if label_indices:
        ax2.scatter(projected[label_indices, 0], projected[label_indices, 1],
                   c=[colors_map[i]], label=CLASSES[label], alpha=0.6, s=50)
ax2.set_title('Embeddings by Semantic Class')
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'embedding_visualization.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Embedding visualization saved")

# ============================================================================
# STAGE 4: GAN GENERATION
# ============================================================================
print("\n" + "="*80)
print("STAGE 4: Image Generation with GANs")
print("="*80)

class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False), nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, nc, 4, 2, 1, bias=False), nn.Tanh()
        )
    def forward(self, input):
        return self.main(input)

class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False), nn.Sigmoid()
        )
    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)

gan_transform = transforms.Compose([
    transforms.Resize((64, 64)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
gan_dataset = FabricDataset(image_paths, labels, gan_transform)
gan_dataloader = DataLoader(gan_dataset, batch_size=min(4, len(image_paths)), shuffle=True)

generator = Generator().to(device)
discriminator = Discriminator().to(device)
criterion = nn.BCELoss()
optimizerG = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerD = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

if len(image_paths) >= 4:
    print("   Training GAN...")
    for epoch in range(3):
        for images, _, _ in tqdm(gan_dataloader, desc=f"GAN Epoch {epoch+1}/3"):
            batch_size = images.size(0)
            images = images.to(device)
            discriminator.zero_grad()
            label = torch.full((batch_size,), 1.0, dtype=torch.float, device=device)
            output = discriminator(images)
            errD_real = criterion(output, label)
            errD_real.backward()
            noise = torch.randn(batch_size, 100, 1, 1, device=device)
            fake = generator(noise)
            label.fill_(0.0)
            output = discriminator(fake.detach())
            errD_fake = criterion(output, label)
            errD_fake.backward()
            optimizerD.step()
            generator.zero_grad()
            label.fill_(1.0)
            output = discriminator(fake)
            errG = criterion(output, label)
            errG.backward()
            optimizerG.step()

print("   Generating synthetic images...")
generator.eval()
generated_images = []
num_generate = min(10, len(image_paths))
with torch.no_grad():
    for _ in range(num_generate):
        noise = torch.randn(1, 100, 1, 1, device=device)
        fake_image = generator(noise)
        fake_image = (fake_image + 1) / 2.0
        fake_image = torch.clamp(fake_image, 0, 1)
        generated_images.append(fake_image.cpu())

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
for i, img_tensor in enumerate(generated_images[:8]):
    img = img_tensor.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Generated {i+1}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gan_generated_sample.png', dpi=150, bbox_inches='tight')
plt.close()

generated_tensors = torch.stack(generated_images)
variance = torch.var(generated_tensors.view(len(generated_images), -1), dim=0).mean().item()
gan_results = {
    'num_generated': len(generated_images), 'diversity_score': float(variance),
    'mode_collapse_detected': variance < 0.01
}
with open(OUTPUT_DIR / 'gan_quality_metrics.json', 'w') as f:
    json.dump(gan_results, f, indent=2)
print(f"✅ Generated {len(generated_images)} images")

# ============================================================================
# STAGE 5: STRESS TESTING
# ============================================================================
print("\n" + "="*80)
print("STAGE 5: Stress Testing with Generated Data")
print("="*80)

vit_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

vit_model.eval()
vit_gen_confidences = []
with torch.no_grad():
    for img_tensor in generated_images:
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)
        img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
        img_tensor_vit = vit_transform(img_pil).unsqueeze(0).to(device)
        output = vit_model(img_tensor_vit)
        probs = torch.softmax(output, dim=1)
        conf, _ = torch.max(probs, 1)
        vit_gen_confidences.append(conf.item())

swin_model.eval()
swin_gen_confidences = []
with torch.no_grad():
    for img_tensor in generated_images:
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)
        img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
        img_tensor_swin = vit_transform(img_pil).unsqueeze(0).to(device)
        output = swin_model(img_tensor_swin)
        probs = torch.softmax(output, dim=1)
        conf, _ = torch.max(probs, 1)
        swin_gen_confidences.append(conf.item())

vit_real_avg_conf = np.mean(vit_confidences)
swin_real_avg_conf = np.mean(swin_confidences)
vit_gen_avg_conf = np.mean(vit_gen_confidences)
swin_gen_avg_conf = np.mean(swin_gen_confidences)

stress_results = {
    'vit': {'real_avg_confidence': float(vit_real_avg_conf), 'generated_avg_confidence': float(vit_gen_avg_conf),
            'confidence_drop': float(vit_real_avg_conf - vit_gen_avg_conf)},
    'swin': {'real_avg_confidence': float(swin_real_avg_conf), 'generated_avg_confidence': float(swin_gen_avg_conf),
             'confidence_drop': float(swin_real_avg_conf - swin_gen_avg_conf)}
}
with open(OUTPUT_DIR / 'cross_model_comparison.json', 'w') as f:
    json.dump(stress_results, f, indent=2)
print(f"✅ Stress testing complete")

# ============================================================================
# STAGE 6: FAILURE ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("STAGE 6: Failure Discovery and Analysis")
print("="*80)

disagreements = []
for i in range(len(image_paths)):
    if vit_predictions[i] != swin_predictions[i]:
        disagreements.append({
            'image_idx': i, 'image_path': image_paths[i], 'true_label': CLASSES[labels[i]],
            'vit_prediction': CLASSES[vit_predictions[i]], 'swin_prediction': CLASSES[swin_predictions[i]],
            'vit_confidence': vit_confidences[i], 'swin_confidence': swin_confidences[i]
        })

clip_sensitive = []
for i in range(len(image_paths)):
    if clip_confidences[i] < 0.3:
        clip_sensitive.append({
            'image_idx': i, 'image_path': image_paths[i], 'true_label': CLASSES[labels[i]],
            'clip_prediction': CLASSES[clip_predictions[i]] if clip_predictions[i] < len(CLASSES) else 'unknown',
            'confidence': clip_confidences[i]
        })

generated_failures = []
for i, conf in enumerate(vit_gen_confidences):
    if conf < 0.2:
        generated_failures.append({'type': 'vit_low_confidence', 'generated_image_idx': i, 'confidence': conf})
for i, conf in enumerate(swin_gen_confidences):
    if conf < 0.2:
        generated_failures.append({'type': 'swin_low_confidence', 'generated_image_idx': i, 'confidence': conf})

limitations = {'vit_limitations': [], 'swin_limitations': [], 'clip_limitations': [], 'gan_limitations': []}
if vit_accuracy < 0.5:
    limitations['vit_limitations'].append('Low accuracy on small dataset')
if vit_avg_confidence < 0.5:
    limitations['vit_limitations'].append('Low prediction confidence')
if swin_accuracy < 0.5:
    limitations['swin_limitations'].append('Low accuracy on small dataset')
if swin_avg_inference_time > 0.1:
    limitations['swin_limitations'].append('Slower inference time')
if clip_accuracy < 0.4:
    limitations['clip_limitations'].append('Low zero-shot accuracy')
if np.mean(clip_confidences) < 0.3:
    limitations['clip_limitations'].append('High prompt sensitivity')
limitations['gan_limitations'].extend([
    'Mode collapse risk with small datasets', 'Training instability',
    'Generated images may not match real distribution'
])

failure_results = {
    'cases': {'vit_swin_disagreements': disagreements[:10], 'clip_prompt_sensitivity': clip_sensitive[:10],
              'generated_image_failures': generated_failures[:10]},
    'vit_limitations': limitations['vit_limitations'], 'swin_limitations': limitations['swin_limitations'],
    'clip_limitations': limitations['clip_limitations'], 'gan_limitations': limitations['gan_limitations'],
    'summary': {'total_disagreements': len(disagreements), 'total_clip_sensitive': len(clip_sensitive),
                'total_generated_failures': len(generated_failures)}
}
with open(OUTPUT_DIR / 'failure_cases' / 'failure_cases.json', 'w') as f:
    json.dump(failure_results, f, indent=2)
print(f"✅ Found {len(disagreements)} disagreements, {len(clip_sensitive)} CLIP sensitive, {len(generated_failures)} generated failures")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("GENERATING FINAL SUMMARY")
print("="*80)

summary = {
    "pipeline_metadata": {"num_images": len(image_paths), "num_classes": len(CLASSES), "device": str(device)},
    "stage1_vision_transformers": {
        "vit_vs_swin": {
            "vit_accuracy": vit_accuracy, "swin_accuracy": swin_accuracy,
            "vit_avg_confidence": float(vit_avg_confidence), "swin_avg_confidence": float(swin_avg_confidence),
            "vit_inference_time": float(vit_avg_inference_time), "swin_inference_time": float(swin_avg_inference_time)
        },
        "conclusion": "ViT and Swin show different strengths in visual understanding"
    },
    "stage2_clip_reasoning": {
        "supervised_vs_zero_shot": {
            "supervised_accuracy": (vit_accuracy + swin_accuracy) / 2, "clip_zero_shot_accuracy": clip_accuracy,
            "clip_success_where_supervised_failed": clip_success_supervised_failed,
            "supervised_success_where_clip_failed": supervised_success_clip_failed
        },
        "conclusion": "CLIP provides complementary reasoning through language supervision"
    },
    "stage4_gan_generation": {"gan_quality_metrics": gan_results, "conclusion": "GAN demonstrates generation capabilities with identified limitations"},
    "stage5_stress_testing": {"gan_impact_on_recognition": stress_results, "conclusion": "Generated images reveal model robustness and generalization gaps"},
    "stage6_failure_analysis": {
        "identified_limitations": limitations,
        "failure_cases_count": len(disagreements) + len(clip_sensitive) + len(generated_failures),
        "conclusion": "Failure cases highlight architectural weaknesses and improvement opportunities"
    },
    "overall_conclusions": {
        "architecture_comparison": "Each architecture (ViT, Swin, CLIP) has unique strengths and weaknesses",
        "generative_modeling": "GANs enable stress testing but reveal training instabilities",
        "cross_architectural_insights": "Embedding alignment reveals semantic understanding differences",
        "recommendations": [
            "Combine supervised and zero-shot approaches for robustness",
            "Use embedding alignment for model ensemble",
            "Address GAN mode collapse for better synthetic data",
            "Leverage failure cases for targeted dataset augmentation"
        ]
    }
}

with open(OUTPUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE")
print("="*80)
print(f"📁 All results saved to: {OUTPUT_DIR}")
print(f"\n📊 Key Results:")
print(f"   ViT Accuracy: {vit_accuracy:.4f}")
print(f"   Swin Accuracy: {swin_accuracy:.4f}")
print(f"   CLIP Zero-shot Accuracy: {clip_accuracy:.4f}")
print(f"   Generated Images: {len(generated_images)}")
print(f"   Failure Cases: {len(disagreements) + len(clip_sensitive) + len(generated_failures)}")
print("\n📁 Output Files:")
print(f"   - vit_results.json, swin_results.json, clip_results.json")
print(f"   - embedding_visualization.png, gan_generated_sample.png")
print(f"   - gan_quality_metrics.json, cross_model_comparison.json")
print(f"   - failure_cases/failure_cases.json, summary.json")


## Stage 1: Vision Transformer-Based Visual Understanding

Train/fine-tune ViT and Swin Transformers for image classification.


In [ ]:
# Stage 1: Vision Transformers
import timm

# Load dataset
DATASET_PATH = KAGGLE_INPUT / 'fabric-final-dataset' / 'FabrIQ_Final_Dataset'
train_dir = DATASET_PATH / 'train' / 'images'

if not train_dir.exists():
    # Try alternative path
    DATASET_PATH = KAGGLE_INPUT / 'fabric' / 'FabrIQ_Final_Dataset'
    train_dir = DATASET_PATH / 'train' / 'images'

print(f"📁 Dataset path: {DATASET_PATH}")
print(f"📁 Train images: {train_dir}")

# Load images
image_paths = []
labels = []
all_images = list(train_dir.glob("*.jpg"))[:NUM_SAMPLES]

for img_path in all_images:
    parts = img_path.stem.split('_')
    if len(parts) >= 2 and parts[0] == 'FabrIQ':
        class_parts = parts[1:-1]
        class_name = ' '.join(class_parts)
        if class_name in CLASSES:
            image_paths.append(str(img_path))
            labels.append(CLASSES.index(class_name))

print(f"✅ Loaded {len(image_paths)} images")

# Dataset class
class FabricDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx], self.image_paths[idx]
        except Exception as e:
            print(f"Error loading {self.image_paths[idx]}: {e}")
            return torch.zeros(3, 224, 224), self.labels[idx], self.image_paths[idx]


In [ ]:
# Train/fine-tune ViT
print("="*80)
print("STAGE 1: Training ViT")
print("="*80)

vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=len(CLASSES))
vit_model = vit_model.to(device)

# Fine-tune if we have enough data
if len(image_paths) >= 4:
    dataset = FabricDataset(image_paths, labels)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(vit_model.parameters(), lr=1e-4, weight_decay=0.01)
    
    vit_model.train()
    for epoch in range(2):  # Quick fine-tuning
        total_loss = 0
        for images, labels_batch, _ in tqdm(dataloader, desc=f"ViT Epoch {epoch+1}/2"):
            images = images.to(device)
            labels_batch = labels_batch.to(device)
            optimizer.zero_grad()
            outputs = vit_model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"   Epoch {epoch+1} loss: {total_loss/len(dataloader):.4f}")

# Evaluate ViT
vit_model.eval()
vit_predictions = []
vit_confidences = []
vit_inference_times = []
correct = 0

dataset = FabricDataset(image_paths, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

with torch.no_grad():
    for images, labels_batch, _ in dataloader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        start_time = time.time()
        outputs = vit_model(images)
        vit_inference_times.append(time.time() - start_time)
        probs = torch.softmax(outputs, dim=1)
        conf, preds = torch.max(probs, 1)
        vit_predictions.extend(preds.cpu().numpy())
        vit_confidences.extend(conf.cpu().numpy())
        correct += (preds == labels_batch).sum().item()

vit_accuracy = correct / len(image_paths) if len(image_paths) > 0 else 0
vit_avg_confidence = np.mean(vit_confidences)
vit_avg_inference_time = np.mean(vit_inference_times)

# Extract ViT embeddings
vit_embeddings = []
with torch.no_grad():
    for images, _, _ in dataloader:
        images = images.to(device)
        try:
            features = vit_model.forward_features(images)
            if len(features.shape) == 4:
                features = features.mean(dim=[2, 3])
            elif len(features.shape) == 3:
                features = features.mean(dim=1)
            vit_embeddings.append(features.cpu().numpy())
        except:
            output = vit_model(images)
            vit_embeddings.append(output.cpu().numpy())

vit_embeddings = np.concatenate(vit_embeddings, axis=0) if vit_embeddings else np.zeros((len(image_paths), 768))

vit_results = {
    'model_name': 'ViT',
    'accuracy': vit_accuracy,
    'avg_confidence': float(vit_avg_confidence),
    'avg_inference_time': float(vit_avg_inference_time),
    'predictions': vit_predictions,
    'confidences': [float(c) for c in vit_confidences]
}

print(f"✅ ViT Accuracy: {vit_accuracy:.4f}")
print(f"✅ ViT Avg Confidence: {vit_avg_confidence:.4f}")
print(f"✅ ViT Avg Inference Time: {vit_avg_inference_time:.4f}s")

with open(OUTPUT_DIR / 'vit_results.json', 'w') as f:
    json.dump(vit_results, f, indent=2)


In [ ]:
# Train/fine-tune Swin Transformer
print("\n" + "="*80)
print("STAGE 1: Training Swin Transformer")
print("="*80)

swin_model = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=len(CLASSES))
swin_model = swin_model.to(device)

# Fine-tune if we have enough data
if len(image_paths) >= 4:
    dataset = FabricDataset(image_paths, labels)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(swin_model.parameters(), lr=1e-4, weight_decay=0.01)
    
    swin_model.train()
    for epoch in range(2):
        total_loss = 0
        for images, labels_batch, _ in tqdm(dataloader, desc=f"Swin Epoch {epoch+1}/2"):
            images = images.to(device)
            labels_batch = labels_batch.to(device)
            optimizer.zero_grad()
            outputs = swin_model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"   Epoch {epoch+1} loss: {total_loss/len(dataloader):.4f}")

# Evaluate Swin
swin_model.eval()
swin_predictions = []
swin_confidences = []
swin_inference_times = []
correct = 0

dataset = FabricDataset(image_paths, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

with torch.no_grad():
    for images, labels_batch, _ in dataloader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        start_time = time.time()
        outputs = swin_model(images)
        swin_inference_times.append(time.time() - start_time)
        probs = torch.softmax(outputs, dim=1)
        conf, preds = torch.max(probs, 1)
        swin_predictions.extend(preds.cpu().numpy())
        swin_confidences.extend(conf.cpu().numpy())
        correct += (preds == labels_batch).sum().item()

swin_accuracy = correct / len(image_paths) if len(image_paths) > 0 else 0
swin_avg_confidence = np.mean(swin_confidences)
swin_avg_inference_time = np.mean(swin_inference_times)

# Extract Swin embeddings
swin_embeddings = []
with torch.no_grad():
    for images, _, _ in dataloader:
        images = images.to(device)
        try:
            features = swin_model.forward_features(images)
            if len(features.shape) == 4:
                features = features.mean(dim=[2, 3])
            elif len(features.shape) == 3:
                features = features.mean(dim=1)
            swin_embeddings.append(features.cpu().numpy())
        except:
            output = swin_model(images)
            swin_embeddings.append(output.cpu().numpy())

swin_embeddings = np.concatenate(swin_embeddings, axis=0) if swin_embeddings else np.zeros((len(image_paths), 768))

swin_results = {
    'model_name': 'Swin',
    'accuracy': swin_accuracy,
    'avg_confidence': float(swin_avg_confidence),
    'avg_inference_time': float(swin_avg_inference_time),
    'predictions': swin_predictions,
    'confidences': [float(c) for c in swin_confidences]
}

print(f"✅ Swin Accuracy: {swin_accuracy:.4f}")
print(f"✅ Swin Avg Confidence: {swin_avg_confidence:.4f}")
print(f"✅ Swin Avg Inference Time: {swin_avg_inference_time:.4f}s")

with open(OUTPUT_DIR / 'swin_results.json', 'w') as f:
    json.dump(swin_results, f, indent=2)


## Stage 2: Vision-Language Reasoning with CLIP

Zero-shot classification and image-text retrieval using CLIP.


In [ ]:
# Stage 2: CLIP Vision-Language Reasoning
print("\n" + "="*80)
print("STAGE 2: CLIP Vision-Language Reasoning")
print("="*80)

import clip

# Load CLIP model
print("   Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("   ✅ CLIP model loaded")

# Zero-shot classification
text_descriptions = [f"a photo of {cls}" for cls in CLASSES]
text_tokens = clip.tokenize(text_descriptions).to(device)

with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

clip_predictions = []
clip_confidences = []
clip_embeddings = []
correct = 0

with torch.no_grad():
    for i, img_path in enumerate(tqdm(image_paths, desc="   CLIP classification")):
        try:
            image = Image.open(img_path).convert('RGB')
            image_tensor = clip_preprocess(image).unsqueeze(0).to(device)
            image_features = clip_model.encode_image(image_tensor)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            clip_embeddings.append(image_features.cpu().numpy())
            
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            conf, pred = similarity.max(dim=-1)
            
            clip_predictions.append(pred.item())
            clip_confidences.append(conf.item())
            
            if pred.item() == labels[i]:
                correct += 1
        except Exception as e:
            print(f"   Error: {e}")
            clip_predictions.append(0)
            clip_confidences.append(0.0)
            clip_embeddings.append(np.zeros((1, 512)))

clip_embeddings = np.concatenate(clip_embeddings, axis=0) if clip_embeddings else np.zeros((len(image_paths), 512))
clip_accuracy = correct / len(image_paths) if len(image_paths) > 0 else 0

# Compare with supervised models
clip_success_supervised_failed = 0
supervised_success_clip_failed = 0

for i in range(len(image_paths)):
    clip_correct = clip_predictions[i] == labels[i]
    vit_correct = vit_predictions[i] == labels[i]
    swin_correct = swin_predictions[i] == labels[i]
    supervised_correct = vit_correct or swin_correct
    
    if clip_correct and not supervised_correct:
        clip_success_supervised_failed += 1
    if supervised_correct and not clip_correct:
        supervised_success_clip_failed += 1

clip_results = {
    'zero_shot_accuracy': clip_accuracy,
    'zero_shot_predictions': clip_predictions,
    'zero_shot_confidences': [float(c) for c in clip_confidences],
    'clip_success_count': clip_success_supervised_failed,
    'supervised_success_count': supervised_success_clip_failed
}

print(f"✅ CLIP Zero-shot Accuracy: {clip_accuracy:.4f}")
print(f"✅ CLIP succeeded where supervised failed: {clip_success_supervised_failed}")
print(f"✅ Supervised succeeded where CLIP failed: {supervised_success_clip_failed}")

with open(OUTPUT_DIR / 'clip_results.json', 'w') as f:
    json.dump(clip_results, f, indent=2)


## Stage 3: Cross-Architectural Embedding Alignment

Project embeddings to shared space and visualize alignment.


In [ ]:
# Stage 3: Embedding Alignment
print("\n" + "="*80)
print("STAGE 3: Cross-Architectural Embedding Alignment")
print("="*80)

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# Normalize embeddings
vit_emb_norm = vit_embeddings / (np.linalg.norm(vit_embeddings, axis=1, keepdims=True) + 1e-8)
swin_emb_norm = swin_embeddings / (np.linalg.norm(swin_embeddings, axis=1, keepdims=True) + 1e-8)
clip_emb_norm = clip_embeddings / (np.linalg.norm(clip_embeddings, axis=1, keepdims=True) + 1e-8)

# Combine embeddings
all_embeddings = np.concatenate([vit_emb_norm, swin_emb_norm, clip_emb_norm], axis=0)
embedding_names = ['vit'] * len(vit_emb_norm) + ['swin'] * len(swin_emb_norm) + ['clip'] * len(clip_emb_norm)

# Project to 2D
print("   Projecting embeddings to 2D...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_embeddings)-1))
projected = tsne.fit_transform(all_embeddings)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot by model
ax1 = axes[0]
colors = {'vit': 'red', 'swin': 'blue', 'clip': 'green'}
for model_name in ['vit', 'swin', 'clip']:
    indices = [i for i, name in enumerate(embedding_names) if name == model_name]
    if indices:
        ax1.scatter(projected[indices, 0], projected[indices, 1], 
                   c=colors.get(model_name, 'gray'), 
                   label=model_name.upper(), alpha=0.6, s=50)
ax1.set_title('Embeddings by Model Architecture')
ax1.set_xlabel('Dimension 1')
ax1.set_ylabel('Dimension 2')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot by class
ax2 = axes[1]
unique_labels = list(set(labels))
colors_map = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))

for i, label in enumerate(unique_labels):
    label_indices = []
    for img_idx, img_label in enumerate(labels):
        if img_label == label:
            for model_name in ['vit', 'swin', 'clip']:
                model_start = sum(len(vit_emb_norm) if m == 'vit' else len(swin_emb_norm) if m == 'swin' else len(clip_emb_norm) 
                                for m in ['vit', 'swin', 'clip'] if m < model_name)
                emb_idx = model_start + img_idx
                if emb_idx < len(projected):
                    label_indices.append(emb_idx)
    
    if label_indices:
        ax2.scatter(projected[label_indices, 0], projected[label_indices, 1],
                   c=[colors_map[i]], label=CLASSES[label], alpha=0.6, s=50)

ax2.set_title('Embeddings by Semantic Class')
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'embedding_visualization.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Embedding visualization saved to {OUTPUT_DIR / 'embedding_visualization.png'}")


## Stage 4: Image Generation with GANs

Train a GAN to generate synthetic images for stress testing.


In [ ]:
# Stage 4: GAN Generation
print("\n" + "="*80)
print("STAGE 4: Image Generation with GANs")
print("="*80)

# Simple GAN Generator and Discriminator
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, input):
        return self.main(input)

class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)

# Prepare dataset for GAN (64x64 images)
gan_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

gan_dataset = FabricDataset(image_paths, labels, gan_transform)
gan_dataloader = DataLoader(gan_dataset, batch_size=min(4, len(image_paths)), shuffle=True)

# Initialize GAN
generator = Generator().to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()
lr = 0.0002
optimizerG = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizerD = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# Train GAN (quick training)
print("   Training GAN...")
real_label = 1.0
fake_label = 0.0

if len(image_paths) >= 4:
    for epoch in range(3):  # Quick training
        for i, (images, _, _) in enumerate(tqdm(gan_dataloader, desc=f"GAN Epoch {epoch+1}/3")):
            batch_size = images.size(0)
            images = images.to(device)
            
            # Train Discriminator
            discriminator.zero_grad()
            label = torch.full((batch_size,), real_label, dtype=torch.float, device=device)
            output = discriminator(images)
            errD_real = criterion(output, label)
            errD_real.backward()
            
            noise = torch.randn(batch_size, 100, 1, 1, device=device)
            fake = generator(noise)
            label.fill_(fake_label)
            output = discriminator(fake.detach())
            errD_fake = criterion(output, label)
            errD_fake.backward()
            errD = errD_real + errD_fake
            optimizerD.step()
            
            # Train Generator
            generator.zero_grad()
            label.fill_(real_label)
            output = discriminator(fake)
            errG = criterion(output, label)
            errG.backward()
            optimizerG.step()

# Generate images
print("   Generating synthetic images...")
generator.eval()
generated_images = []
num_generate = min(10, len(image_paths))

with torch.no_grad():
    for _ in range(num_generate):
        noise = torch.randn(1, 100, 1, 1, device=device)
        fake_image = generator(noise)
        fake_image = (fake_image + 1) / 2.0
        fake_image = torch.clamp(fake_image, 0, 1)
        generated_images.append(fake_image.cpu())

# Save sample
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
for i, img_tensor in enumerate(generated_images[:8]):
    img = img_tensor.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Generated {i+1}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gan_generated_sample.png', dpi=150, bbox_inches='tight')
plt.close()

# Evaluate GAN quality
generated_tensors = torch.stack(generated_images)
variance = torch.var(generated_tensors.view(len(generated_images), -1), dim=0).mean().item()

gan_results = {
    'num_generated': len(generated_images),
    'diversity_score': float(variance),
    'mode_collapse_detected': variance < 0.01
}

print(f"✅ Generated {len(generated_images)} images")
print(f"✅ Diversity score: {variance:.4f}")
print(f"✅ GAN samples saved to {OUTPUT_DIR / 'gan_generated_sample.png'}")

with open(OUTPUT_DIR / 'gan_quality_metrics.json', 'w') as f:
    json.dump(gan_results, f, indent=2)


In [ ]:
# Stage 5: Stress Testing
print("\n" + "="*80)
print("STAGE 5: Stress Testing with Generated Data")
print("="*80)

# Test models on generated images
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test ViT on generated
vit_model.eval()
vit_gen_confidences = []
with torch.no_grad():
    for img_tensor in generated_images:
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)
        img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
        img_tensor_vit = vit_transform(img_pil).unsqueeze(0).to(device)
        output = vit_model(img_tensor_vit)
        probs = torch.softmax(output, dim=1)
        conf, _ = torch.max(probs, 1)
        vit_gen_confidences.append(conf.item())

# Test Swin on generated
swin_model.eval()
swin_gen_confidences = []
with torch.no_grad():
    for img_tensor in generated_images:
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)
        img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
        img_tensor_swin = vit_transform(img_pil).unsqueeze(0).to(device)
        output = swin_model(img_tensor_swin)
        probs = torch.softmax(output, dim=1)
        conf, _ = torch.max(probs, 1)
        swin_gen_confidences.append(conf.item())

# Compare with real images
vit_real_avg_conf = np.mean(vit_confidences)
swin_real_avg_conf = np.mean(swin_confidences)
vit_gen_avg_conf = np.mean(vit_gen_confidences)
swin_gen_avg_conf = np.mean(swin_gen_confidences)

stress_results = {
    'vit': {
        'real_avg_confidence': float(vit_real_avg_conf),
        'generated_avg_confidence': float(vit_gen_avg_conf),
        'confidence_drop': float(vit_real_avg_conf - vit_gen_avg_conf)
    },
    'swin': {
        'real_avg_confidence': float(swin_real_avg_conf),
        'generated_avg_confidence': float(swin_gen_avg_conf),
        'confidence_drop': float(swin_real_avg_conf - swin_gen_avg_conf)
    }
}

print(f"✅ ViT: Real conf={vit_real_avg_conf:.4f}, Generated conf={vit_gen_avg_conf:.4f}, Drop={vit_real_avg_conf - vit_gen_avg_conf:.4f}")
print(f"✅ Swin: Real conf={swin_real_avg_conf:.4f}, Generated conf={swin_gen_avg_conf:.4f}, Drop={swin_real_avg_conf - swin_gen_avg_conf:.4f}")

with open(OUTPUT_DIR / 'cross_model_comparison.json', 'w') as f:
    json.dump(stress_results, f, indent=2)


## Stage 6: Failure Discovery and Analysis

Identify and analyze failure cases across all models.


In [ ]:
# Stage 6: Failure Analysis
print("\n" + "="*80)
print("STAGE 6: Failure Discovery and Analysis")
print("="*80)

# Find ViT-Swin disagreements
disagreements = []
for i in range(len(image_paths)):
    if vit_predictions[i] != swin_predictions[i]:
        disagreements.append({
            'image_idx': i,
            'image_path': image_paths[i],
            'true_label': CLASSES[labels[i]],
            'vit_prediction': CLASSES[vit_predictions[i]],
            'swin_prediction': CLASSES[swin_predictions[i]],
            'vit_confidence': vit_confidences[i],
            'swin_confidence': swin_confidences[i]
        })

# Find CLIP low confidence cases
clip_sensitive = []
for i in range(len(image_paths)):
    if clip_confidences[i] < 0.3:
        clip_sensitive.append({
            'image_idx': i,
            'image_path': image_paths[i],
            'true_label': CLASSES[labels[i]],
            'clip_prediction': CLASSES[clip_predictions[i]] if clip_predictions[i] < len(CLASSES) else 'unknown',
            'confidence': clip_confidences[i]
        })

# Find generated image failures
generated_failures = []
for i, conf in enumerate(vit_gen_confidences):
    if conf < 0.2:
        generated_failures.append({
            'type': 'vit_low_confidence',
            'generated_image_idx': i,
            'confidence': conf
        })
for i, conf in enumerate(swin_gen_confidences):
    if conf < 0.2:
        generated_failures.append({
            'type': 'swin_low_confidence',
            'generated_image_idx': i,
            'confidence': conf
        })

# Identify limitations
limitations = {
    'vit_limitations': [],
    'swin_limitations': [],
    'clip_limitations': [],
    'gan_limitations': []
}

if vit_accuracy < 0.5:
    limitations['vit_limitations'].append('Low accuracy on small dataset')
if vit_avg_confidence < 0.5:
    limitations['vit_limitations'].append('Low prediction confidence')

if swin_accuracy < 0.5:
    limitations['swin_limitations'].append('Low accuracy on small dataset')
if swin_avg_inference_time > 0.1:
    limitations['swin_limitations'].append('Slower inference time')

if clip_accuracy < 0.4:
    limitations['clip_limitations'].append('Low zero-shot accuracy')
if np.mean(clip_confidences) < 0.3:
    limitations['clip_limitations'].append('High prompt sensitivity')

limitations['gan_limitations'].extend([
    'Mode collapse risk with small datasets',
    'Training instability',
    'Generated images may not match real distribution'
])

failure_results = {
    'cases': {
        'vit_swin_disagreements': disagreements[:10],
        'clip_prompt_sensitivity': clip_sensitive[:10],
        'generated_image_failures': generated_failures[:10]
    },
    'vit_limitations': limitations['vit_limitations'],
    'swin_limitations': limitations['swin_limitations'],
    'clip_limitations': limitations['clip_limitations'],
    'gan_limitations': limitations['gan_limitations'],
    'summary': {
        'total_disagreements': len(disagreements),
        'total_clip_sensitive': len(clip_sensitive),
        'total_generated_failures': len(generated_failures)
    }
}

print(f"✅ Found {len(disagreements)} ViT-Swin disagreements")
print(f"✅ Found {len(clip_sensitive)} CLIP sensitive cases")
print(f"✅ Found {len(generated_failures)} generated image failures")

with open(OUTPUT_DIR / 'failure_cases' / 'failure_cases.json', 'w') as f:
    json.dump(failure_results, f, indent=2)


## Final Summary

Generate comprehensive summary of all pipeline results.


In [ ]:
# Generate Final Summary
print("\n" + "="*80)
print("GENERATING FINAL SUMMARY")
print("="*80)

summary = {
    "pipeline_metadata": {
        "num_images": len(image_paths),
        "num_classes": len(CLASSES),
        "device": str(device)
    },
    "stage1_vision_transformers": {
        "vit_vs_swin": {
            "vit_accuracy": vit_accuracy,
            "swin_accuracy": swin_accuracy,
            "vit_avg_confidence": float(vit_avg_confidence),
            "swin_avg_confidence": float(swin_avg_confidence),
            "vit_inference_time": float(vit_avg_inference_time),
            "swin_inference_time": float(swin_avg_inference_time)
        },
        "conclusion": "ViT and Swin show different strengths in visual understanding"
    },
    "stage2_clip_reasoning": {
        "supervised_vs_zero_shot": {
            "supervised_accuracy": (vit_accuracy + swin_accuracy) / 2,
            "clip_zero_shot_accuracy": clip_accuracy,
            "clip_success_where_supervised_failed": clip_success_supervised_failed,
            "supervised_success_where_clip_failed": supervised_success_clip_failed
        },
        "conclusion": "CLIP provides complementary reasoning through language supervision"
    },
    "stage4_gan_generation": {
        "gan_quality_metrics": gan_results,
        "conclusion": "GAN demonstrates generation capabilities with identified limitations"
    },
    "stage5_stress_testing": {
        "gan_impact_on_recognition": stress_results,
        "conclusion": "Generated images reveal model robustness and generalization gaps"
    },
    "stage6_failure_analysis": {
        "identified_limitations": limitations,
        "failure_cases_count": len(disagreements) + len(clip_sensitive) + len(generated_failures),
        "conclusion": "Failure cases highlight architectural weaknesses and improvement opportunities"
    },
    "overall_conclusions": {
        "architecture_comparison": "Each architecture (ViT, Swin, CLIP) has unique strengths and weaknesses",
        "generative_modeling": "GANs enable stress testing but reveal training instabilities",
        "cross_architectural_insights": "Embedding alignment reveals semantic understanding differences",
        "recommendations": [
            "Combine supervised and zero-shot approaches for robustness",
            "Use embedding alignment for model ensemble",
            "Address GAN mode collapse for better synthetic data",
            "Leverage failure cases for targeted dataset augmentation"
        ]
    }
}

with open(OUTPUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE")
print("="*80)
print(f"📁 All results saved to: {OUTPUT_DIR}")
print(f"\n📊 Key Results:")
print(f"   ViT Accuracy: {vit_accuracy:.4f}")
print(f"   Swin Accuracy: {swin_accuracy:.4f}")
print(f"   CLIP Zero-shot Accuracy: {clip_accuracy:.4f}")
print(f"   Generated Images: {len(generated_images)}")
print(f"   Failure Cases: {len(disagreements) + len(clip_sensitive) + len(generated_failures)}")
print("\n📁 Output Files:")
print(f"   - vit_results.json")
print(f"   - swin_results.json")
print(f"   - clip_results.json")
print(f"   - embedding_visualization.png")
print(f"   - gan_generated_sample.png")
print(f"   - gan_quality_metrics.json")
print(f"   - cross_model_comparison.json")
print(f"   - failure_cases/failure_cases.json")
print(f"   - summary.json")
